# 08 — Exploration metadata Iceberg

**Objectif** : comprendre la structure interne d'une table Iceberg —
lier chaque fichier Parquet à son entrée dans les manifests  
**Source** : `silver.trays` sur ADLS Gen2  
**Catalog** : SQLite local `catalog_dev.db

In [1]:
import os
import json
import warnings
from pathlib import Path

import pandas as pd
from pyiceberg.catalog.sql import SqlCatalog
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

load_dotenv()

print("Ok")

Ok


In [2]:
import sys
sys.path.append("../src")
from catalog import get_catalog, ADLS_URI
from schema import get_or_create_silver_trays

catalog = get_catalog()
table   = get_or_create_silver_trays(catalog, ADLS_URI)

Table 'silver.trays' chargée — 180 snapshots.
  Location : abfss://silver@dlsecatcandlingfrcedev.dfs.core.windows.net/trays_iceberg
  Format   : Iceberg v2
  Champs   : 27


In [3]:
snap = table.current_snapshot()

print("Snapshot courant ")
print(f"  snapshot_id      : {snap.snapshot_id}")
print(f"  timestamp_ms     : {snap.timestamp_ms}")
print(f"  operation        : {snap.summary.get('operation')}")
print(f"  added-files      : {snap.summary.get('added-data-files')}")
print(f"  added-records    : {snap.summary.get('added-records')}")
print(f"  total-records    : {snap.summary.get('total-records')}")
print(f"  total-data-files : {snap.summary.get('total-data-files')}")
print(f"  total-size-bytes : {int(snap.summary.get('total-files-size', 0)) / 1024 / 1024:.2f} Mo")

Snapshot courant 
  snapshot_id      : 6637810770123150361
  timestamp_ms     : 1782311667884
  operation        : Operation.APPEND
  added-files      : 1
  added-records    : 3
  total-records    : 1344
  total-data-files : 180
  total-size-bytes : 2.69 Mo


In [4]:
history = []
for s in table.snapshots():
    history.append({
        "snapshot_id":   s.snapshot_id,
        "timestamp_ms":  pd.to_datetime(s.timestamp_ms, unit="ms", utc=True),
        "operation":     s.summary.get("operation"),
        "added_files":   s.summary.get("added-data-files", 0),
        "added_records": s.summary.get("added-records", 0),
        "total_records": s.summary.get("total-records", 0),
        "total_files":   s.summary.get("total-data-files", 0),
    })

df_history = pd.DataFrame(history)
print(f"Nombre total de snapshots : {len(df_history)}")
df_history.head(10)

Nombre total de snapshots : 180


,snapshot_id,timestamp_ms,operation,added_files,added_records,total_records,total_files
0,3686287724106525192,2026-06-24 14:32:42.801000+00:00,Operation.APPEND,1,2,2,1
1,8813609995095537524,2026-06-24 14:32:43.394000+00:00,Operation.APPEND,1,11,13,2
2,5270281244342531051,2026-06-24 14:32:43.898000+00:00,Operation.APPEND,1,9,22,3
3,2633451521130558220,2026-06-24 14:32:44.379000+00:00,Operation.APPEND,1,6,28,4
4,8173246843929870836,2026-06-24 14:32:44.839000+00:00,Operation.APPEND,1,4,32,5
5,891038821813661921,2026-06-24 14:32:45.294000+00:00,Operation.APPEND,1,9,41,6
6,8878300186691725336,2026-06-24 14:32:45.853000+00:00,Operation.APPEND,1,10,51,7
7,8385436421705579161,2026-06-24 14:32:46.441000+00:00,Operation.APPEND,1,10,61,8
8,58684751662899023,2026-06-24 14:32:46.892000+00:00,Operation.APPEND,1,9,70,9
9,329825600385903564,2026-06-24 14:32:47.419000+00:00,Operation.APPEND,1,9,79,10


In [5]:
# PyIceberg expose les data files via table.inspect
# C'est l'API officielle pour lister les fichiers physiques

df_files = table.inspect.data_files().to_pandas()

print(f"Nombre de fichiers Parquet : {len(df_files)}")
print(f"Colonnes disponibles : {list(df_files.columns)}")
df_files.head(5)

Nombre de fichiers Parquet : 180
Colonnes disponibles : ['content', 'file_path', 'file_format', 'spec_id', 'partition', 'record_count', 'file_size_in_bytes', 'column_sizes', 'value_counts', 'null_value_counts', 'nan_value_counts', 'lower_bounds', 'upper_bounds', 'key_metadata', 'split_offsets', 'equality_ids', 'sort_order_id', 'readable_metrics']


,content,file_path,file_format,spec_id,partition,record_count,file_size_in_bytes,column_sizes,value_counts,null_value_counts,nan_value_counts,lower_bounds,upper_bounds,key_metadata,split_offsets,equality_ids,sort_order_id,readable_metrics
0,0,abfss://silver@dlsecatcandlingfrcedev.dfs.core...,PARQUET,0,"{'machine_id': 'PMAF-C012501', 'year': 2026, '...",3,12895,"[(1, 317), (2, 102), (3, 123), (4, 86), (5, 86...","[(1, 3), (2, 3), (3, 3), (4, 3), (5, 3), (6, 3...","[(1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0...",[],"[(1, b'274d85ce134448b1'), (2, b'PMAF-C012501'...","[(1, b'ef58033d024d24f2'), (2, b'PMAF-C012501'...",None,[4],None,NaN,"{'tray_id': {'column_size': 317, 'value_count'..."
1,0,abfss://silver@dlsecatcandlingfrcedev.dfs.core...,PARQUET,0,"{'machine_id': 'PMAF-C012501', 'year': 2026, '...",7,15306,"[(1, 470), (2, 102), (3, 153), (4, 86), (5, 86...","[(1, 7), (2, 7), (3, 7), (4, 7), (5, 7), (6, 7...","[(1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0...",[],"[(1, b'098fc9085e9537d7'), (2, b'PMAF-C012501'...","[(1, b'fe4c6ee98fed6a41'), (2, b'PMAF-C012501'...",None,[4],None,NaN,"{'tray_id': {'column_size': 470, 'value_count'..."
2,0,abfss://silver@dlsecatcandlingfrcedev.dfs.core...,PARQUET,0,"{'machine_id': 'PMAF-C012501', 'year': 2026, '...",1,11511,"[(1, 248), (2, 102), (3, 106), (4, 86), (5, 86...","[(1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1...","[(1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0...",[],"[(1, b'5777a9e4972da55d'), (2, b'PMAF-C012501'...","[(1, b'5777a9e4972da55e'), (2, b'PMAF-C012501'...",None,[4],None,NaN,"{'tray_id': {'column_size': 248, 'value_count'..."
3,0,abfss://silver@dlsecatcandlingfrcedev.dfs.core...,PARQUET,0,"{'machine_id': 'PMAF-C012501', 'year': 2026, '...",8,15780,"[(1, 504), (2, 102), (3, 153), (4, 86), (5, 90...","[(1, 8), (2, 8), (3, 8), (4, 8), (5, 8), (6, 8...","[(1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0...",[],"[(1, b'16bfc08f5c2620c2'), (2, b'PMAF-C012501'...","[(1, b'd79ba4f5ebbff445'), (2, b'PMAF-C012501'...",None,[4],None,NaN,"{'tray_id': {'column_size': 504, 'value_count'..."
4,0,abfss://silver@dlsecatcandlingfrcedev.dfs.core...,PARQUET,0,"{'machine_id': 'PMAF-C012501', 'year': 2026, '...",7,15273,"[(1, 466), (2, 102), (3, 150), (4, 86), (5, 86...","[(1, 7), (2, 7), (3, 7), (4, 7), (5, 7), (6, 7...","[(1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0...",[],"[(1, b'2e2aa6436ad133e5'), (2, b'PMAF-C012501'...","[(1, b'd7392425d4f57b1b'), (2, b'PMAF-C012501'...",None,[4],None,NaN,"{'tray_id': {'column_size': 466, 'value_count'..."


In [6]:
# Sélectionner les colonnes utiles
cols = [
    "content", "file_path", "file_format",
    "record_count", "file_size_in_bytes",
    "column_sizes", "value_counts",
    "null_value_counts", "lower_bounds", "upper_bounds",
]

cols_present = [c for c in cols if c in df_files.columns]
df_detail = df_files[cols_present].copy()

# Convertir la taille en Ko
df_detail["file_size_kb"] = (df_detail["file_size_in_bytes"] / 1024).round(1)

# Extraire le nom de fichier court
df_detail["file_name"] = df_detail["file_path"].apply(
    lambda p: "/".join(str(p).split("/")[-3:])  # partition/date/fichier
)

print(f"Total fichiers    : {len(df_detail)}")
print(f"Total records     : {df_detail['record_count'].sum()}")
print(f"Taille totale     : {df_detail['file_size_in_bytes'].sum() / 1024 / 1024:.2f} Mo")
print(f"Taille moy/fichier: {df_detail['file_size_kb'].mean():.1f} Ko")
print(f"Records moy/fichier: {df_detail['record_count'].mean():.1f}")
print()
df_detail[["file_name", "record_count", "file_size_kb"]].head(10)

Total fichiers    : 180
Total records     : 1344
Taille totale     : 2.69 Mo
Taille moy/fichier: 15.3 Ko
Records moy/fichier: 7.5



,file_name,record_count,file_size_kb
0,month=5/day=15/00000-0-fc9546b7-d0a9-43c3-aff7...,3,12.6
1,month=5/day=15/00000-0-17a1a475-9d64-4257-8559...,7,14.9
2,month=5/day=15/00000-0-004c7045-0018-47e0-923d...,1,11.2
3,month=5/day=15/00000-0-de410568-e912-4571-8ed3...,8,15.4
4,month=5/day=15/00000-0-865c13fd-8285-4f0d-9abe...,7,14.9
5,month=5/day=15/00000-0-7d4b5d4e-08b9-4c37-b010...,9,16.4
6,month=5/day=15/00000-0-a5205cb8-38ef-425b-9bd2...,4,13.1
7,month=5/day=15/00000-0-d0afae59-0caf-407e-98be...,6,14.4
8,month=5/day=15/00000-0-c4d3a6ec-6a5d-4f5f-9fa0...,10,16.9
9,month=5/day=15/00000-0-346df883-d7a9-4d7e-8608...,9,16.4


In [10]:
manifests_data = []
for snap in table.snapshots():
    for m in snap.manifests(table.io):
        manifests_data.append({
            "snapshot_id":    snap.snapshot_id,
            "snapshot_ts":    pd.to_datetime(snap.timestamp_ms, unit="ms", utc=True),
            "manifest_path":  m.manifest_path,
            "added_files":    m.added_files_count,
            "existing_files": m.existing_files_count,
            "deleted_files":  m.deleted_files_count,
        })

df_manifests = pd.DataFrame(manifests_data)
print(f"Nombre de manifests : {len(df_manifests)}")
df_manifests.head(10)

Nombre de manifests : 16290


,snapshot_id,snapshot_ts,manifest_path,added_files,existing_files,deleted_files
0,3686287724106525192,2026-06-24 14:32:42.801000+00:00,abfss://silver@dlsecatcandlingfrcedev.dfs.core...,1,0,0
1,8813609995095537524,2026-06-24 14:32:43.394000+00:00,abfss://silver@dlsecatcandlingfrcedev.dfs.core...,1,0,0
2,8813609995095537524,2026-06-24 14:32:43.394000+00:00,abfss://silver@dlsecatcandlingfrcedev.dfs.core...,1,0,0
3,5270281244342531051,2026-06-24 14:32:43.898000+00:00,abfss://silver@dlsecatcandlingfrcedev.dfs.core...,1,0,0
4,5270281244342531051,2026-06-24 14:32:43.898000+00:00,abfss://silver@dlsecatcandlingfrcedev.dfs.core...,1,0,0
5,5270281244342531051,2026-06-24 14:32:43.898000+00:00,abfss://silver@dlsecatcandlingfrcedev.dfs.core...,1,0,0
6,2633451521130558220,2026-06-24 14:32:44.379000+00:00,abfss://silver@dlsecatcandlingfrcedev.dfs.core...,1,0,0
7,2633451521130558220,2026-06-24 14:32:44.379000+00:00,abfss://silver@dlsecatcandlingfrcedev.dfs.core...,1,0,0
8,2633451521130558220,2026-06-24 14:32:44.379000+00:00,abfss://silver@dlsecatcandlingfrcedev.dfs.core...,1,0,0
9,2633451521130558220,2026-06-24 14:32:44.379000+00:00,abfss://silver@dlsecatcandlingfrcedev.dfs.core...,1,0,0


In [11]:
# Extraire partition depuis le chemin du fichier directement
df_files["machine_id"]   = df_files["file_path"].apply(
    lambda p: str(p).split("machine_id=")[1].split("/")[0] 
    if "machine_id=" in str(p) else None
)
df_files["candled_date"] = df_files["file_path"].apply(
    lambda p: str(p).split("candled_date=")[1].split("/")[0] 
    if "candled_date=" in str(p) else None
)

df_partitions = (
    df_files
    .groupby(["machine_id", "candled_date"])
    .agg(
        nb_fichiers   = ("file_path",          "count"),
        total_records = ("record_count",        "sum"),
        total_size_kb = ("file_size_in_bytes",  lambda x: round(x.sum() / 1024, 1))
    )
    .reset_index()
    .sort_values("total_records", ascending=False)
)

print(f"Partitions distinctes : {len(df_partitions)}")
print(f"Total fichiers        : {df_partitions['nb_fichiers'].sum()}")
print(f"Total records         : {df_partitions['total_records'].sum()}")
print(f"Total size            : {df_partitions['total_size_kb'].sum() / 1024:.2f} Mo")
print()
df_partitions

Partitions distinctes : 0
Total fichiers        : 0
Total records         : 0
Total size            : 0.00 Mo



,machine_id,candled_date,nb_fichiers,total_records,total_size_kb


In [12]:
print("=== Distribution des tailles de fichiers ===")
print(f"  Min     : {df_detail['file_size_kb'].min():.1f} Ko")
print(f"  Médiane : {df_detail['file_size_kb'].median():.1f} Ko")
print(f"  Moy     : {df_detail['file_size_kb'].mean():.1f} Ko")
print(f"  Max     : {df_detail['file_size_kb'].max():.1f} Ko")
print()

# Contexte : production max 7h/jour
# 7h × 60 min = 420 fichiers/jour/couvoir
HEURES_PRODUCTION    = 7
FICHIERS_PAR_JOUR    = HEURES_PRODUCTION * 60
JOURS_PAR_MOIS       = 20  # jours de production réels en moyenne

nb_fichiers          = len(df_detail)
records_par_fichier  = df_detail["record_count"].mean()

print(f"Fichiers actuels         : {nb_fichiers}")
print(f"Records moy par fichier  : {records_par_fichier:.1f}")
print()
print(f"=== Projection prod (1 couvoir) ===")
print(f"  Fichiers/jour    : ~{FICHIERS_PAR_JOUR}")
print(f"  Fichiers/mois    : ~{FICHIERS_PAR_JOUR * JOURS_PAR_MOIS}")
print(f"  Fichiers/an      : ~{FICHIERS_PAR_JOUR * JOURS_PAR_MOIS * 12}")
print()

=== Distribution des tailles de fichiers ===
  Min     : 11.2 Ko
  Médiane : 16.3 Ko
  Moy     : 15.3 Ko
  Max     : 17.5 Ko

Fichiers actuels         : 180
Records moy par fichier  : 7.5

=== Projection prod (1 couvoir) ===
  Fichiers/jour    : ~420
  Fichiers/mois    : ~8400
  Fichiers/an      : ~100800



In [ ]:
import json
from pyiceberg.io.pyarrow import PyArrowFileIO

# Lire le fichier metadata JSON courant depuis ADLS Gen2
io = table.io
metadata_location = table.metadata_location

print(f"metadata_location : {metadata_location}\n")

with io.new_input(metadata_location).open() as f:
    meta_raw = json.load(f)

print(f"format-version   : {meta_raw.get('format-version')}")
print(f"table-uuid       : {meta_raw.get('table-uuid')}")
print(f"location         : {meta_raw.get('location')}")
print(f"last-updated-ms  : {pd.to_datetime(meta_raw.get('last-updated-ms'), unit='ms', utc=True)}")
print(f"current-snapshot-id : {meta_raw.get('current-snapshot-id')}")
print(f"nb schemas       : {len(meta_raw.get('schemas', []))}")
print(f"nb snapshots     : {len(meta_raw.get('snapshots', []))}")
print(f"nb partition-specs : {len(meta_raw.get('partition-specs', []))}")

metadata_location : abfss://silver@dlsecatcandlingfrcedev.dfs.core.windows.net/trays_iceberg/metadata/00180-cf678c8b-c418-4b54-9b99-7e23f42f3ad9.metadata.json

format-version   : 2
table-uuid       : a61e577a-f8ae-420f-ba51-7a8edc516e89
location         : abfss://silver@dlsecatcandlingfrcedev.dfs.core.windows.net/trays_iceberg
last-updated-ms  : 2026-06-24 14:34:27.884000+00:00
current-snapshot-id : 6637810770123150361
nb schemas       : 1
nb snapshots     : 180
nb partition-specs : 1


: 